# Notebook 7 — Fine-tuning : LoRA & RLHF
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Panorama du fine-tuning](#overview)
2. [Full Fine-Tuning](#full)
3. [PEFT : Parameter-Efficient Fine-Tuning](#peft)
4. [LoRA (Low-Rank Adaptation)](#lora)
5. [QLoRA — LoRA quantifié](#qlora)
6. [RLHF (Reinforcement Learning from Human Feedback)](#rlhf)
7. [DPO (Direct Preference Optimization)](#dpo)
8. [Instruction Tuning & Prompt Engineering](#instruct)
9. [Questions d’entretien](#questions)


---
## 1. Panorama du fine-tuning <a name='overview'></a>

### Du pre-entrainement au deploiement

```
Pre-training (CLM/MLM)        -> Modele de base (base model)
    |
    v
Instruction Tuning (SFT)      -> Modele instruit (suit des instructions)
    |
    v
RLHF / DPO / RLAIF            -> Modele aligne (utile, inoffensif, honnete)
    |
    v
Fine-tuning specifique (LoRA) -> Modele specialise (domaine, langue, tache)
```

### Strategies de fine-tuning

| Strategie | Params modifies | GPU requis | Performance |
|---|---|---|---|
| Full fine-tuning | 100% | Tres eleve (80GB+) | Meilleure |
| Adapter layers | ~0.5-3% | Moyen | Tres bonne |
| LoRA | 0.1-1% | Faible | Bonne |
| QLoRA | 0.1-1% | Minimal (4-bit) | Bonne |
| Prefix tuning | ~0.1% | Faible | Bonne sur NLP |
| Prompt tuning | <0.01% | Minimal | Variable |
| Zero-shot / ICL | 0% | Inference seule | Depend du modele |


In [ ]:
import numpy as np

# ============================================================
# COMPARAISON : FULL FT vs PEFT
# ============================================================
print("=== Memoire requise pour le fine-tuning ===")
print()

def estimate_ft_memory(n_params_B, precision="bf16", with_optimizer=True):
    """
    Estimation de la memoire GPU pour le fine-tuning
    precision: bf16 (2 bytes) ou fp32 (4 bytes) ou int4 (0.5 bytes)
    with_optimizer: Adam (2 copies des gradients)
    """
    bytes_per_param = {"fp32": 4, "bf16": 2, "fp16": 2, "int4": 0.5}[precision]
    n = n_params_B * 1e9

    model_mem  = n * bytes_per_param
    grad_mem   = n * 4  # gradients toujours en fp32
    optim_mem  = n * 8  # Adam: m (fp32) + v (fp32)

    if with_optimizer:
        total = model_mem + grad_mem + optim_mem
    else:
        total = model_mem

    return total / 1e9  # en GB

print(f"{'Modele':15s}  {'Params':>8}  {'Full FT':>10}  {'LoRA (1%)':>10}  {'QLoRA (4bit)':>14}")
models = [
    ("LLaMA 2 7B",  7),
    ("LLaMA 2 13B", 13),
    ("LLaMA 2 70B", 70),
    ("Mistral 7B",  7),
]
for name, p in models:
    full  = estimate_ft_memory(p, "bf16", True)
    lora_model = p * 2 / 1e0   # modele bf16 seulement + 1% de params trainables
    lora_extra = p * 0.01 * (4 + 8 + 4) / 1e0  # 1% params: grad+optim en fp32
    lora_total = (p * 2 + p * 0.01 * 16) / 1e0   # approximation
    qlora = (p * 0.5 + p * 0.01 * 16) / 1e0   # modele 4-bit + lora fp32

    print(f"  {name:13s}  {p:>6}B  {full:>8.0f} GB  {lora_total:>8.0f} GB  {qlora:>12.0f} GB")

print("\n  Note: QLoRA permet de faire tourner LLaMA 2 70B sur un seul A100 80GB!")


---
## 2. Full Fine-Tuning <a name='full'></a>

### Principe
Mettre a jour **tous** les parametres du modele sur un dataset specifique.

### Objectif
Minimiser la Cross-Entropy Loss sur les donnees de fine-tuning :
```
L = -1/N * sum log P_theta(y_t | y_<t, x)
```

### Risques
- **Catastrophic forgetting** : le modele oublie ses connaissances generales
- **Overfitting** : sur de petits datasets
- Solutions : weight decay, early stopping, learning rate faible (1e-5 a 1e-4)

### Bonnes pratiques
- Commencer par geler les premieres couches (les plus generiques)
- Utiliser un lr schedule avec warmup
- Surveiller la validation loss a chaque epoch
- Gradient clipping (max_norm=1.0)

### Catastrophic Forgetting
- Probleme bien connu en apprentissage continu
- Solutions : Elastic Weight Consolidation (EWC), Replay, LoRA (n'affecte pas W_0)


In [ ]:
import numpy as np

# ============================================================
# SIMULATION DU CATASTROPHIC FORGETTING
# ============================================================
class SimpleModel:
    def __init__(self, n_in, n_out, seed=42):
        np.random.seed(seed)
        self.W = np.random.randn(n_out, n_in) * 0.1
        self.b = np.zeros(n_out)

    def forward(self, x):
        return x @ self.W.T + self.b

    def loss(self, x, y):
        pred = self.forward(x)
        return np.mean((pred - y)**2)

    def update(self, x, y, lr=0.01):
        pred = self.forward(x)
        err  = pred - y
        dW = 2 * err.T @ x / len(x)
        db = 2 * err.mean(axis=0)
        self.W -= lr * dW
        self.b -= lr * db

np.random.seed(0)

# Tache A : classification de formes (pre-training)
X_A = np.random.randn(100, 4)
y_A = X_A @ np.array([[1.0, 0.5, -0.3, 0.2],
                        [-0.2, 1.0, 0.5, -0.1]])

# Tache B : nouvelle domaine (fine-tuning)
X_B = np.random.randn(50, 4)
y_B = X_B @ np.array([[-0.5, 1.2, 0.0, 0.8],
                        [0.9, -0.3, 1.0, -0.5]])

model_full = SimpleModel(4, 2)

# Phase 1: apprendre tache A
for _ in range(200):
    model_full.update(X_A, y_A, lr=0.01)
loss_A_before = model_full.loss(X_A, y_A)

# Phase 2: fine-tuning sur tache B (sans protection)
for _ in range(200):
    model_full.update(X_B, y_B, lr=0.01)
loss_A_after = model_full.loss(X_A, y_A)
loss_B_after = model_full.loss(X_B, y_B)

print("=== Catastrophic Forgetting ===")
print(f"  Loss tache A avant fine-tuning : {loss_A_before:.4f}")
print(f"  Loss tache A apres fine-tuning : {loss_A_after:.4f}  (devrait etre plus grande!)")
print(f"  Loss tache B apres fine-tuning : {loss_B_after:.4f}")
print(f"  Degradation tache A            : {(loss_A_after - loss_A_before) / loss_A_before * 100:.1f}%")

# ============================================================
# EWC (Elastic Weight Consolidation) -- protection
# ============================================================
print("\n=== Elastic Weight Consolidation (EWC) ===")
print("""
  Apres avoir appris la tache A, on calcule la Fisher Information Matrix:
    F_i = E[(dlogP/dw_i)^2]
  Elle mesure l importance de chaque poids pour la tache A.

  Lors du fine-tuning sur B, on ajoute une regularisation:
    L_EWC = L_B + lambda * sum_i F_i * (w_i - w_A_i)^2

  => Les poids importants pour A sont penalises s ils changent trop.
  => Equilibre entre memorisation (tache A) et adaptation (tache B).
""")

# Simulation simple de EWC
class SimpleModelEWC(SimpleModel):
    def __init__(self, n_in, n_out, seed=42):
        super().__init__(n_in, n_out, seed)
        self.fisher = None
        self.W_A    = None

    def compute_fisher(self, x, y, n_samples=50):
        """Fisher Information: variance du gradient"""
        idx = np.random.choice(len(x), n_samples, replace=False)
        x_s, y_s = x[idx], y[idx]
        pred = self.forward(x_s)
        err  = pred - y_s
        grad_W = 2 * err.T @ x_s / n_samples
        self.fisher = grad_W**2
        self.W_A    = self.W.copy()

    def update_ewc(self, x, y, lr=0.01, lam=10.0):
        pred = self.forward(x)
        err  = pred - y
        dW = 2 * err.T @ x / len(x)
        db = 2 * err.mean(axis=0)
        # Regularisation EWC
        if self.fisher is not None:
            dW += lam * self.fisher * (self.W - self.W_A)
        self.W -= lr * dW
        self.b -= lr * db

model_ewc = SimpleModelEWC(4, 2)
for _ in range(200):
    model_ewc.update(X_A, y_A, lr=0.01)

loss_ewc_A_before = model_ewc.loss(X_A, y_A)
model_ewc.compute_fisher(X_A, y_A)

for _ in range(200):
    model_ewc.update_ewc(X_B, y_B, lr=0.01, lam=5.0)

loss_ewc_A_after = model_ewc.loss(X_A, y_A)
loss_ewc_B_after = model_ewc.loss(X_B, y_B)
print(f"  Loss tache A avant (EWC) : {loss_ewc_A_before:.4f}")
print(f"  Loss tache A apres (EWC) : {loss_ewc_A_after:.4f}")
print(f"  Loss tache B apres (EWC) : {loss_ewc_B_after:.4f}")
print(f"  Degradation (EWC)        : {(loss_ewc_A_after - loss_ewc_A_before) / loss_ewc_A_before * 100:.1f}%  (reduite!)")


---
## 3. PEFT : Parameter-Efficient Fine-Tuning <a name='peft'></a>

### Idee generale
Geler les poids originaux W_0 et n'entrainer qu'un **petit nombre de parametres additionnels**.

### Methodes principales

**Adapter Layers (Houlsby 2019)**
- Inserer de petites couches (bottleneck FFN) entre les couches Transformer
- Down-project (d -> r) -> activation -> Up-project (r -> d)
- Params additionnels : 2 * d * r par couche

**Prefix Tuning (Li & Liang 2021)**
- Ajouter des "soft tokens" virtuels en debut de sequence (dans K et V)
- Ces tokens sont appris, le reste du modele est gele
- Params : n_prefix * d_model * n_layers

**Prompt Tuning (Lester 2021)**
- Cas extreme : seulement des tokens virtuels en input
- Pas de modification des couches internes

**LoRA (Hu 2021)**
- Decomposer la mise a jour DeltaW en produit de deux matrices de bas rang
- Voir section suivante


In [ ]:
import numpy as np

# ============================================================
# ADAPTER LAYERS -- from scratch
# ============================================================
class AdapterLayer:
    """
    Houlsby Adapter: bottleneck FFN insere apres MHA et FFN
    down: d -> r (reduction)
    up:   r -> d (expansion)
    Residuel: x + up(activation(down(x)))
    """
    def __init__(self, d_model, r=8):
        self.d = d_model
        self.r = r
        # Initialisation proche de l identite
        scale = np.sqrt(2.0 / d_model)
        self.W_down = np.random.randn(r, d_model) * scale * 0.01
        self.b_down = np.zeros(r)
        self.W_up   = np.random.randn(d_model, r) * scale * 0.01
        self.b_up   = np.zeros(d_model)

    def forward(self, x):
        h = np.tanh(x @ self.W_down.T + self.b_down)
        out = h @ self.W_up.T + self.b_up
        return x + out  # residuel!

    @property
    def n_params(self):
        return 2 * self.d * self.r + self.r + self.d

np.random.seed(42)
d_model = 768   # BERT-base
seq_len, batch = 10, 2

for r in [4, 8, 16, 64]:
    adapter = AdapterLayer(d_model, r)
    x = np.random.randn(batch, seq_len, d_model)
    out = adapter.forward(x)
    ratio = adapter.n_params / (d_model * d_model) * 100

    print(f"Adapter r={r:3d}: params={adapter.n_params:,}  "
          f"vs Dense({d_model}x{d_model})={d_model*d_model:,}  "
          f"ratio={ratio:.1f}%  out={out.shape}")

# ============================================================
# CALCUL DU NB DE PARAMS PEFT
# ============================================================
print("\n=== Comparaison du nb de params PEFT pour BERT-large ===")
d_model = 1024
n_layers = 24
n_heads  = 16
vocab    = 30522
total_bert = 340_000_000  # params BERT-large

methods = {
    "Full FT":          total_bert,
    "Adapter (r=8)":    n_layers * 2 * (2 * d_model * 8),
    "Adapter (r=64)":   n_layers * 2 * (2 * d_model * 64),
    "Prefix (n=20)":    n_layers * 2 * 20 * d_model,   # prefixes dans K et V
    "LoRA (r=4)":       n_layers * 4 * (d_model * 4 + 4 * d_model),   # 4 matrices par couche
    "LoRA (r=16)":      n_layers * 4 * (d_model * 16 + 16 * d_model),
    "Prompt tuning":    20 * d_model,
}
print(f"  {'Methode':20s}  {'Params':>12}  {'% du total':>10}")
for name, p in methods.items():
    print(f"  {name:20s}  {p:>12,}  {p/total_bert*100:>9.3f}%")


---
## 4. LoRA (Low-Rank Adaptation) <a name='lora'></a>

### Hypothese fondamentale
Les **mises a jour de poids** lors du fine-tuning ont une faible **rang intrinseque**.
=> On peut les approcher par un produit de deux matrices de bas rang.

### Formule
```
W' = W_0 + DeltaW = W_0 + B * A
```
Ou :
- `W_0 in R^(d x k)` : poids originaux **geles**
- `A in R^(r x k)` : matrice down-projection (init: gaussienne)
- `B in R^(d x r)` : matrice up-projection (init: **zeros**)
- `r << min(d, k)` : rang (hyperparametre)

### Forward pass
```
h = W_0 * x + (B * A) * x * (alpha / r)
```
- `alpha / r` : facteur de scaling (alpha souvent egal a r => scaling = 1)
- Initialiser B=0 => au debut DeltaW=0 => debut identique au modele original

### Ou appliquer LoRA ?
- **W_Q et W_V** seulement (papier original)
- **W_Q, W_K, W_V, W_O** (meilleurs resultats en pratique)
- **W_Q, W_K, W_V, W_O, W_up, W_gate, W_down** (full LoRA)

### Rang r
- r=4 a 16 : tres efficace pour la plupart des taches
- r=64 : pour des taches plus complexes ou peu de data
- r=256+ : approche le full fine-tuning

### Merge des poids (inference)
Au deploiement, on peut **fusionner** LoRA dans W_0 :
```
W' = W_0 + B * A   (une seule matrice, zero overhead)
```


In [ ]:
import numpy as np

# ============================================================
# LORA -- implementation from scratch
# ============================================================
class LoRALinear:
    """
    Couche lineaire avec LoRA:
    output = x @ W_0.T + x @ (B @ A).T * (alpha/r)
    """
    def __init__(self, d_in, d_out, r=8, alpha=16, seed=42):
        np.random.seed(seed)
        self.d_in  = d_in
        self.d_out = d_out
        self.r     = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Poids originaux -- GELES
        self.W_0 = np.random.randn(d_out, d_in) * np.sqrt(2.0 / d_in)
        self.b   = np.zeros(d_out)

        # LoRA matrices -- ENTRAINABLES
        self.A = np.random.randn(r, d_in)  * np.sqrt(2.0 / d_in)
        self.B = np.zeros((d_out, r))       # init a zero!

    def forward(self, x):
        # Output original (W_0 gele)
        h0 = x @ self.W_0.T + self.b
        # Output LoRA (B @ A gele seulement au debut)
        lora_out = (x @ self.A.T) @ self.B.T * self.scaling
        return h0 + lora_out

    def merge(self):
        """Fusionner LoRA dans W_0 pour l inference (zero overhead)"""
        W_merged = self.W_0 + self.B @ self.A * self.scaling
        return W_merged

    @property
    def n_params_lora(self):
        return self.r * (self.d_in + self.d_out)

    @property
    def n_params_full(self):
        return self.d_in * self.d_out

    @property
    def compression_ratio(self):
        return self.n_params_lora / self.n_params_full

np.random.seed(42)
d_in, d_out = 768, 768   # attention de BERT-base

print("=== LoRA : impact du rang r ===")
print(f"{'Rang r':>8}  {'Params LoRA':>12}  {'Params Full':>12}  {'Ratio':>8}  {'Output ok'}")
for r in [1, 2, 4, 8, 16, 64, 256]:
    lora = LoRALinear(d_in, d_out, r=r, alpha=r)
    x = np.random.randn(4, 10, d_in)   # batch=4, seq=10
    out = lora.forward(x)
    print(f"  {r:6d}  {lora.n_params_lora:>12,}  {lora.n_params_full:>12,}  "
          f"{lora.compression_ratio:>7.2%}  {out.shape}")

print()
print("=== Initialisation B=0 : pourquoi ? ===")
# Au debut de l entrainement, DeltaW = B @ A = 0 @ A = 0
# => Le modele LoRA est identique au modele original au step 0
lora0 = LoRALinear(d_in, d_out, r=8)
x = np.random.randn(2, 5, d_in)
out_original = x @ lora0.W_0.T + lora0.b
out_lora     = lora0.forward(x)
diff = np.abs(out_original - out_lora).max()
print(f"  Max diff initial (B=0): {diff:.2e}  (doit etre ~0)")
print("  => La formation commence exactement comme le modele de base")


In [ ]:
import numpy as np

# ============================================================
# LORA : ENTRAINEMENT SIMPLIFIE
# ============================================================
class LoRALinear:
    def __init__(self, d_in, d_out, r=8, alpha=16, seed=42):
        np.random.seed(seed)
        self.d_in, self.d_out, self.r = d_in, d_out, r
        self.scaling = alpha / r
        self.W_0 = np.random.randn(d_out, d_in) * np.sqrt(2.0 / d_in)
        self.b   = np.zeros(d_out)
        self.A   = np.random.randn(r, d_in) * np.sqrt(2.0 / d_in)
        self.B   = np.zeros((d_out, r))
        # Gradients accumules
        self.dA = np.zeros_like(self.A)
        self.dB = np.zeros_like(self.B)

    def forward(self, x):
        self._x = x
        h0 = x @ self.W_0.T + self.b
        self._Ax = x @ self.A.T
        return h0 + self._Ax @ self.B.T * self.scaling

    def backward(self, dout):
        # dout : (batch, seq, d_out)
        x, Ax = self._x, self._Ax
        n = x.shape[0] * x.shape[1]
        # Gradients LoRA (W_0 est gele)
        self.dB = (dout * self.scaling).reshape(-1, self.d_out).T @ Ax.reshape(-1, self.r) / n
        dAx = dout @ self.B * self.scaling
        self.dA = dAx.reshape(-1, self.r).T @ x.reshape(-1, self.d_in) / n
        return dout @ self.W_0  # gradient vers la couche precedente

    def update(self, lr=1e-4):
        self.A -= lr * self.dA
        self.B -= lr * self.dB

    def merge(self):
        return self.W_0 + self.B @ self.A * self.scaling

    @property
    def n_params_lora(self):
        return self.r * (self.d_in + self.d_out)

# Simulation d un fine-tuning LoRA sur une tache de classification simple
np.random.seed(42)
d_in, d_out, r = 32, 16, 4
batch, seq = 8, 5

lora = LoRALinear(d_in, d_out, r=r, alpha=r)

X = np.random.randn(batch, seq, d_in)
Y = np.random.randn(batch, seq, d_out)   # targets

print("=== Fine-tuning LoRA (simulation) ===")
losses = []
for step in range(200):
    out  = lora.forward(X)
    loss = np.mean((out - Y)**2)
    losses.append(loss)
    dl = 2 * (out - Y) / (batch * seq)
    lora.backward(dl)
    lora.update(lr=1e-3)

print(f"Loss initiale : {losses[0]:.6f}")
print(f"Loss finale   : {losses[-1]:.6f}")
print(f"Reduction     : {(1 - losses[-1]/losses[0])*100:.1f}%")
print()
# Norme des updates LoRA
print(f"Norme W_0 (gele)    : {np.linalg.norm(lora.W_0):.4f}  (inchange)")
print(f"Norme A (appris)    : {np.linalg.norm(lora.A):.4f}")
print(f"Norme B (appris)    : {np.linalg.norm(lora.B):.4f}")
print(f"Norme DeltaW = B@A  : {np.linalg.norm(lora.B @ lora.A * lora.scaling):.4f}")

# Merge et comparaison
W_merged = lora.merge()
out_merged = X.reshape(-1, d_in) @ W_merged.T + lora.b
out_lora   = lora.forward(X).reshape(-1, d_out)
diff = np.abs(out_merged - out_lora).max()
print(f"\nMax diff apres merge : {diff:.2e}  (identique!)")
print("=> On peut fusionner B@A dans W_0 sans perte => zero overhead a l inference")


In [ ]:
import numpy as np

# ============================================================
# LORA APPLIQUE A L ATTENTION -- comment ca marche en pratique
# ============================================================
print("=== LoRA sur les couches d attention ===")
print("""
Papier LoRA original (Hu 2021):
  Applique LoRA sur W_Q et W_V seulement
  r=4, alpha=4 pour GPT-3 => 0.01% des params!

Pratique courante:
  Applique sur W_Q, W_K, W_V, W_O (parfois aussi FFN)
  r=8 a 64 selon la complexite de la tache

Configuration typique par modele:
""")
configs = [
    ("GPT-3 175B (papier)", "Q, V",         4,   4,    "0.01%",  "SuperGLUE ~= full FT"),
    ("LLaMA 2 7B (chat)",   "Q, K, V, O",   8,   16,   "0.08%",  "Instruction following"),
    ("LLaMA 2 7B (code)",   "All (Q,K,V,O,ffn)", 16, 32, "0.3%", "Code generation"),
    ("BERT (NER)",          "Q, V",         8,   8,    "0.5%",   "NER, classification"),
]
print(f"  {'Modele':25s}  {'Cibles':20s}  {'r':>4}  {'alpha':>6}  {'Params%':>8}  {'Tache'}")
for name, target, r, alpha, pct, task in configs:
    print(f"  {name:25s}  {target:20s}  {r:>4}  {alpha:>6}  {pct:>8}  {task}")

print("""
Hyperparametres importants:
  r      : rang. Plus grand => plus de capacite, plus de params.
            Commencer avec r=8, augmenter si sous-performance.
  alpha  : scaling factor. En pratique on fixe alpha=r (scaling=1)
            ou alpha=2r (scaling=2). L adapter lentement est conseille.
  target : Q+V minimum, Q+K+V+O meilleur. Inclure FFN si tache complexe.
  lr     : typiquement 2e-4 a 1e-3 (plus eleve que full FT car moins de params)
  dropout: 0.05 a 0.1 apres les matrices A pour regulariser
""")


---
## 5. QLoRA — LoRA quantifié <a name='qlora'></a>

### Innovation (Dettmers et al., 2023)
Combine deux techniques :
1. **Quantification NF4** : stocker W_0 en 4 bits au lieu de 16 bits => 4x moins de memoire
2. **Double quantification** : quantifier aussi les constantes de quantification
3. **Paged Optimizers** : gerer les pics memoire via la pagination GPU/CPU
4. **LoRA** : entrainer uniquement les matrices A et B en BF16

### Quantification NF4 (NormalFloat4)
- Concu pour les poids de reseaux de neurones (distribution normale)
- 16 valeurs de quantification optimalement reparties sur N(0, 1)
- Meilleur que INT4 pour les poids neuraux

### Workflow QLoRA
```
W_0 (fp16/bf16)
  -> Quantifier en NF4    (stockage)
  -> Dequantifier en bf16 (calcul forward)
  -> Calculer h = W_0_dequant * x + B @ A * x
  -> Gradient vers A et B seulement
  -> Mettre a jour A et B en bf16
```

### Resultats
- LLaMA 65B finetunable sur un seul GPU A100 80GB
- Seulement ~1-2% de degradation vs full fine-tuning 16-bit


In [ ]:
import numpy as np

# ============================================================
# QUANTIFICATION : INT8, NF4, FP4 -- principes
# ============================================================
def quantize_absmax_int8(tensor):
    """
    Quantification INT8 absmax:
    q = round(x / max(|x|) * 127)
    """
    scale = np.max(np.abs(tensor))
    q = np.round(tensor / scale * 127).astype(np.int8)
    return q, scale

def dequantize_int8(q, scale):
    return q.astype(np.float32) * scale / 127

def quantize_nf4(tensor):
    """
    NF4: 16 niveaux de quantification optimaux pour N(0,1)
    Calculés comme les quantiles de la distribution normale.
    """
    # Les 16 niveaux NF4 (approximation)
    nf4_levels = np.array([
        -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848,
        -0.0911, 0.0, 0.0796, 0.1609, 0.2461, 0.3379,
        0.4407, 0.5626, 0.7230, 1.0
    ], dtype=np.float32)

    # Normaliser dans [-1, 1]
    scale = np.max(np.abs(tensor))
    x_norm = tensor / (scale + 1e-10)

    # Assigner le niveau NF4 le plus proche
    diffs = np.abs(x_norm[..., np.newaxis] - nf4_levels)
    indices = np.argmin(diffs, axis=-1).astype(np.uint8)  # 4 bits!
    return indices, scale, nf4_levels

def dequantize_nf4(indices, scale, nf4_levels):
    return nf4_levels[indices] * scale

# Comparison sur un vecteur de poids
np.random.seed(42)
weights = np.random.randn(64).astype(np.float32)   # poids typiques

q8, s8        = quantize_absmax_int8(weights)
w8            = dequantize_int8(q8, s8)
idx4, s4, lvl = quantize_nf4(weights)
w4            = dequantize_nf4(idx4, s4, lvl)

err_int8 = np.abs(weights - w8).mean()
err_nf4  = np.abs(weights - w4).mean()
snr_int8 = 10 * np.log10(np.var(weights) / np.var(weights - w8))
snr_nf4  = 10 * np.log10(np.var(weights) / np.var(weights - w4))

print("=== Comparaison des quantifications ===")
print(f"  Poids originaux (FP32) : {weights[:6].round(4)}")
print(f"  INT8 dequant           : {w8[:6].round(4)}")
print(f"  NF4  dequant           : {w4[:6].round(4)}")
print()
print(f"  Erreur moyenne INT8    : {err_int8:.6f}")
print(f"  Erreur moyenne NF4     : {err_nf4:.6f}")
print(f"  SNR INT8               : {snr_int8:.2f} dB")
print(f"  SNR NF4                : {snr_nf4:.2f} dB")
print()
print("=== Economie memoire ===")
n = 7_000_000_000  # LLaMA 7B
print(f"  LLaMA 2 7B:")
print(f"  FP32  : {n * 4 / 1e9:.1f} GB")
print(f"  BF16  : {n * 2 / 1e9:.1f} GB")
print(f"  INT8  : {n * 1 / 1e9:.1f} GB")
print(f"  NF4   : {n * 0.5 / 1e9:.1f} GB   (4 bits = 0.5 bytes)")
print(f"  NF4 + LoRA trainable params (1%): +{n * 0.01 * 2 / 1e9:.2f} GB (BF16)")
print(f"  => QLoRA total estim.: {n * 0.5 / 1e9 + n * 0.01 * 2 / 1e9 + 5:.1f} GB  (avec activations etc.)")


---
## 6. RLHF (Reinforcement Learning from Human Feedback) <a name='rlhf'></a>

### Pipeline RLHF (InstructGPT, ChatGPT)

**Etape 1 : Supervised Fine-Tuning (SFT)**
- Fine-tuner le modele sur des demonstrations humaines de haute qualite
- `(prompt, reponse_ideale)` => cross-entropy loss classique

**Etape 2 : Reward Model Training**
- Collecter des comparaisons humaines : `reponse_A` prefere a `reponse_B`
- Entrainer un modele de recompense : `r(prompt, reponse)` -> score scalaire
- Loss : `L = -E[log sigma(r(prompt, y_w) - r(prompt, y_l))]`
  - y_w = reponse preferee (winner), y_l = reponse rejetee (loser)
  - Maximiser la marge entre les deux

**Etape 3 : PPO (Proximal Policy Optimization)**
- Optimiser le modele SFT en utilisant le reward model comme recompense
- Contrainte KL pour ne pas trop s'eloigner du modele SFT
- `reward = r(prompt, y) - beta * KL(pi_RL || pi_SFT)`

### Problemes de RLHF
- Complexe a implementer (3 modeles en memoire : policy, reward, SFT ref)
- Instable (PPO tres sensible aux hyperparametres)
- Reward hacking : le modele optimise le reward sans reellement s'ameliorer


In [ ]:
import numpy as np

# ============================================================
# REWARD MODEL -- from scratch
# ============================================================
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class RewardModel:
    """
    Modele de recompense: prend (prompt, reponse) -> score scalaire
    Entraine sur des paires (prefere, rejete) avec la BT loss
    """
    def __init__(self, d_model, seed=42):
        np.random.seed(seed)
        self.W = np.random.randn(d_model) * 0.01
        self.b = 0.0
        self.lr = 0.01

    def score(self, hidden):
        """hidden : (d_model,) -> scalar"""
        return float(np.dot(self.W, hidden) + self.b)

    def bradley_terry_loss(self, h_w, h_l):
        """
        Loss Bradley-Terry:
        L = -log(sigma(r_w - r_l))
        Maximise la marge entre reponse preferee et rejetee
        """
        r_w = self.score(h_w)
        r_l = self.score(h_l)
        loss = -np.log(sigmoid(r_w - r_l) + 1e-10)
        # Gradient
        sig  = sigmoid(r_w - r_l)
        grad = (sig - 1) * (h_w - h_l)
        return loss, grad

    def update(self, h_w, h_l):
        loss, grad = self.bradley_terry_loss(h_w, h_l)
        self.W -= self.lr * grad
        return loss

np.random.seed(42)
d_model = 16

# Simuler des paires (hidden_state_preferee, hidden_state_rejetee)
# Reponses preferes ont des features positives, rejetees negatives
n_pairs = 200
h_w_all = np.random.randn(n_pairs, d_model) + np.array([0.3] * d_model)
h_l_all = np.random.randn(n_pairs, d_model) + np.array([-0.3] * d_model)

rm = RewardModel(d_model)
losses = []
for i in range(n_pairs):
    loss = rm.update(h_w_all[i], h_l_all[i])
    losses.append(loss)

print("=== Reward Model Training ===")
print(f"Loss initiale : {losses[0]:.4f}")
print(f"Loss finale   : {np.mean(losses[-20:]):.4f}")

# Test: le RM doit scorer les preferes plus haut
test_w = np.random.randn(20, d_model) + 0.3
test_l = np.random.randn(20, d_model) - 0.3
scores_w = [rm.score(h) for h in test_w]
scores_l = [rm.score(h) for h in test_l]
accuracy = np.mean([sw > sl for sw, sl in zip(scores_w, scores_l)])
print(f"Accuracy (r_w > r_l) : {accuracy:.2%}")
print(f"Score moyen prefere  : {np.mean(scores_w):.4f}")
print(f"Score moyen rejete   : {np.mean(scores_l):.4f}")


In [ ]:
import numpy as np

# ============================================================
# PPO SIMPLIFIE -- principes cles
# ============================================================
print("=== PPO (Proximal Policy Optimization) en RLHF ===")
print("""
Objectif PPO en RLHF:
  max E[min(r_t * A_t, clip(r_t, 1-eps, 1+eps) * A_t)]

  r_t = pi_RL(a_t|s_t) / pi_old(a_t|s_t)  (ratio de probabilites)
  A_t = avantage (reward - baseline)
  eps = 0.2                                (hyperparametre de clipping)

Objectif total avec contrainte KL:
  L = L_PPO + lambda * L_KL + c * L_value

Ou:
  L_KL = KL(pi_RL || pi_SFT)  (regularisation: ne pas trop s eloigner)
  L_value = MSE(V(s_t), target)  (baseline du critic)
  beta ~ 0.02 a 0.2              (coefficient KL)
""")

def ppo_clip_loss(old_log_prob, new_log_prob, advantage, eps=0.2):
    """
    PPO clipped surrogate objective
    """
    ratio = np.exp(new_log_prob - old_log_prob)  # r_t
    clipped = np.clip(ratio, 1 - eps, 1 + eps)
    loss = -np.minimum(ratio * advantage, clipped * advantage)
    return loss.mean(), ratio.mean()

# Simulation
np.random.seed(42)
n = 100
old_log_prob = np.random.randn(n) * 0.5
advantage = np.random.randn(n)

print("=== Effet du clipping PPO ===")
print(f"  {'Nouveau log_prob':25s}  {'Loss':>8}  {'Ratio moyen':>12}  {'Clipping'}")
for noise in [0.0, 0.1, 0.3, 0.7, 1.5]:
    new_log_prob = old_log_prob + np.random.randn(n) * noise
    loss, ratio = ppo_clip_loss(old_log_prob, new_log_prob, advantage)
    clipped_frac = np.mean(
        np.abs(np.exp(new_log_prob - old_log_prob) - 1) > 0.2
    )
    print(f"  sigma={noise:.1f} ({n} samples)     {loss:>8.4f}  {ratio:>12.4f}  {clipped_frac:.1%}")

print("\n=> Le clipping empeche des mises a jour trop grandes")
print("   Si ratio > 1+eps ou < 1-eps: gradient = 0 (pas de maj)")

# ============================================================
# KL DIVERGENCE -- contrainte dans RLHF
# ============================================================
print("\n=== Contrainte KL dans RLHF ===")
print("""
  reward_total = r_theta(prompt, response) - beta * KL(pi_RL || pi_SFT)

  KL = sum_t log pi_RL(y_t) - log pi_SFT(y_t)

  Beta typique: 0.01 a 0.1
  - Beta trop petit: reward hacking (le modele triche)
  - Beta trop grand: reste trop proche du SFT, pas assez d amelioration

Reward hacking typique:
  - Reponses tres longues (verboses scoring mieux)
  - Repetitions de phrases qui scorent bien
  - "Sycophantic" responses (dire ce que l humain veut entendre)
  Solution: diverse reward ensemble, constitutional AI, RLAIF
""")


---
## 7. DPO (Direct Preference Optimization) <a name='dpo'></a>

### Motivation
RLHF est complexe et instable. DPO reformule le probleme directement.

### Formule
```
L_DPO = -E[log sigma(beta * log(pi_theta(y_w|x)/pi_ref(y_w|x))
                   - beta * log(pi_theta(y_l|x)/pi_ref(y_l|x)))]
```

### Intuition
- `pi_ref` : le modele SFT (reference gelee)
- Augmenter la probabilite des reponses preferees (`y_w`)
- Diminuer la probabilite des reponses rejetees (`y_l`)
- Beta controle la regularisation (rester proche de pi_ref)

### Avantages vs RLHF
- Pas besoin de reward model separee
- Pas de PPO (algorithme instable)
- Entrainement simple : une seule passe de gradient
- Equivalent mathematiquement a RLHF sous certaines hypotheses

### Variantes recentes
| Methode | Innovation |
|---|---|
| DPO | Original, stable |
| IPO (Identity PO) | Corrige overfitting de DPO |
| KTO | Utilise des donnees non appariees (non paires) |
| ORPO | Pas de modele de reference |
| SimPO | Reference implicite via longueur |


In [ ]:
import numpy as np

# ============================================================
# DPO -- implementation from scratch
# ============================================================
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class DPOTrainer:
    """
    Direct Preference Optimization (Rafailov et al., 2023)
    Entraine directement sur des paires (preferee, rejetee)
    sans reward model ni PPO.
    """
    def __init__(self, beta=0.1):
        self.beta = beta

    def dpo_loss(self, log_prob_theta_w, log_prob_theta_l,
                       log_prob_ref_w,   log_prob_ref_l):
        """
        log_prob_theta : log P_theta(reponse | prompt) = sum log P(token_i)
        log_prob_ref   : log P_ref(reponse | prompt)   (modele SFT gele)
        """
        # Log-ratio: combien le modele theta prefere y_w vs la ref
        log_ratio_w = log_prob_theta_w - log_prob_ref_w
        log_ratio_l = log_prob_theta_l - log_prob_ref_l

        # DPO loss: margin entre w et l
        margin = self.beta * (log_ratio_w - log_ratio_l)
        loss = -np.log(sigmoid(margin) + 1e-10)
        return loss.mean(), margin

    def compute_log_prob(self, logits, token_ids):
        """
        logits    : (batch, seq, vocab)
        token_ids : (batch, seq)
        => log P(sequence) = sum log P(token_i)
        """
        batch, seq, vocab = logits.shape
        log_probs = []
        for b in range(batch):
            lp = 0.0
            for t in range(seq):
                e = np.exp(logits[b, t] - logits[b, t].max())
                p = e / e.sum()
                lp += np.log(p[token_ids[b, t]] + 1e-10)
            log_probs.append(lp)
        return np.array(log_probs)

np.random.seed(42)
dpo = DPOTrainer(beta=0.1)

# Simulation : modele theta (en cours d entrainement) vs modele ref (SFT)
n_pairs = 50
batch_size = 8
vocab_size = 100
seq_len = 10

# Cas 1: debut de l entrainement (theta = ref)
log_ratio_w_0 = np.zeros(n_pairs)   # theta = ref => ratio = 0
log_ratio_l_0 = np.zeros(n_pairs)
loss0, margin0 = dpo.dpo_loss(log_ratio_w_0, log_ratio_l_0,
                               np.zeros(n_pairs), np.zeros(n_pairs))

# Cas 2: modele bien aligne (prefere w > l)
log_ratio_w_good = np.random.randn(n_pairs) * 0.5 + 0.5   # theta prefere w
log_ratio_l_good = np.random.randn(n_pairs) * 0.5 - 0.5   # theta rejette l
loss_good, margin_good = dpo.dpo_loss(
    log_ratio_w_good, log_ratio_l_good,
    np.zeros(n_pairs), np.zeros(n_pairs)
)

print("=== DPO Loss ===")
print(f"  Loss debut d entrainement (theta=ref)  : {loss0:.4f}")
print(f"  Loss modele bien aligne                : {loss_good:.4f}  (plus faible)")
print(f"  Margin debut   : {margin0.mean():.4f}")
print(f"  Margin aligne  : {margin_good.mean():.4f}")

print("\n=== DPO vs RLHF ===")
comparison = [
    ("RLHF (InstructGPT)", "Optimal", "Tres complexe (3 modeles)", "ChatGPT, GPT-4"),
    ("DPO",                "Tres bon", "Simple (1 modele)", "LLaMA-2-chat, Zephyr"),
    ("IPO",                "Bon",     "Simple + regularisation", "Gemma-based"),
    ("KTO",                "Bon",     "Donnees non appariees",   "Plusieurs RLHF-free"),
    ("ORPO",               "Bon",     "Pas de modele ref",       "Orpo-Llama"),
]
print(f"  {'Methode':25s}  {'Qualite':12}  {'Complexite':25s}  {'Exemple'}")
for m, q, c, ex in comparison:
    print(f"  {m:25s}  {q:12}  {c:25s}  {ex}")


---
## 8. Instruction Tuning & Prompt Engineering <a name='instruct'></a>

### Instruction Tuning (SFT)
Fine-tuner sur des paires `(instruction, reponse)` de haute qualite.

**Formats de donnees courants :**
```
# Alpaca format
{"instruction": "Traduis en francais:", "input": "Hello world", "output": "Bonjour monde"}

# ChatML format (OpenAI)
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Comment allez-vous ?<|im_end|>
<|im_start|>assistant
Je vais tres bien, merci !<|im_end|>

# LLaMA 2 format
[INST] <<SYS>>
System prompt.
<</SYS>>
User message [/INST]
```

### Loss masquee (instruction masking)
Calculer la loss **uniquement sur les tokens de reponse** (pas sur le prompt) :
```python
labels = [-100, -100, ..., token_1, token_2, ...]  # -100 = ignore
```

### Datasets d'instruction tuning
| Dataset | Taille | Type |
|---|---|---|
| Alpaca | 52k | GPT-4 generated |
| FLAN | 1.8M | Multi-task NLP |
| OpenHermes | 1M | Curated synthetic |
| UltraChat | 1.5M | Multi-turn chat |
| ShareGPT | 90k | Real ChatGPT conversations |


In [ ]:
import numpy as np

# ============================================================
# INSTRUCTION MASKING -- loss sur la reponse uniquement
# ============================================================
print("=== Instruction Masking pour SFT ===")
print()

# Exemple de tokenisation avec masque
instruction = "[INST] Quelle est la capitale de la France ? [/INST]"
response     = "La capitale de la France est Paris."

# Simulation: tokens numeriques
inst_tokens = [1, 25580, 29962, 29871, 12037, 1563, 29871, 1058, 29892, 29871, 2]
resp_tokens = [29871, 2803, 5081, 29901, 3681, 29889, 2]

# Labels: -100 pour l instruction (ignore dans la loss), vrais tokens pour la reponse
all_tokens = inst_tokens + resp_tokens
labels     = [-100] * len(inst_tokens) + resp_tokens

def cross_entropy_with_mask(logits, labels, vocab_size=32000):
    """Loss CE avec masque (-100 = ignore)"""
    total_loss = 0.0
    n_active = 0
    for t, label in enumerate(labels):
        if label == -100:
            continue  # ignore
        e = np.exp(logits[t] - logits[t].max())
        p = e / e.sum()
        total_loss += -np.log(p[label % vocab_size] + 1e-10)
        n_active += 1
    return total_loss / n_active if n_active > 0 else 0.0

np.random.seed(42)
T = len(all_tokens)
vocab_size = 32000
logits = np.random.randn(T, vocab_size)

loss_masked   = cross_entropy_with_mask(logits, labels, vocab_size)
labels_full   = all_tokens  # sans masque
loss_unmasked = cross_entropy_with_mask(logits, labels_full, vocab_size)

print(f"Tokens totaux    : {T} ({len(inst_tokens)} instruction + {len(resp_tokens)} reponse)")
print(f"Labels masques   : {labels}")
print(f"")
print(f"Loss avec masque (reponse seule) : {loss_masked:.4f}")
print(f"Loss sans masque (tout tokenize) : {loss_unmasked:.4f}")
print(f"=> Le masquage evite que le modele optimise les tokens de l instruction")
print(f"   (le prompt est fixe, seule la reponse doit etre apprise)")

# ============================================================
# FORMATS DE PROMPT -- comparaison
# ============================================================
print("\n=== Formats de prompts par modele ===")
formats = {
    "Alpaca": """
Below is an instruction. Write a response.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}""",

    "LLaMA 2 Chat": """
[INST] <<SYS>>
{system}
<</SYS>>

{user_message} [/INST]""",

    "ChatML (OpenAI)": """
<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{user}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>""",

    "Mistral/LLaMA 3": """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{response}<|eot_id|>"""
}
for name, template in formats.items():
    print(f"\n  --- {name} ---")
    print(template[:200] + ("..." if len(template) > 200 else ""))


---
## 9. Questions d’entretien <a name='questions'></a>

**Q : Expliquer LoRA en 30 secondes.**
> On gele W_0 et on ajoute DeltaW = B@A, ou B (d x r) et A (r x k) avec r tres petit. Le forward devient h = W_0*x + B@A*x. On n'entraine que A et B => 0.1-1% des params. A l'inference, on peut merger B@A dans W_0 sans overhead.

**Q : Pourquoi initialiser B a zero dans LoRA ?**
> Pour que DeltaW = B@A = 0 au debut. Le modele LoRA est donc identique au modele de base au step 0, ce qui garantit un demarrage stable et evite les discontinuites.

**Q : Quelle est la difference entre LoRA et QLoRA ?**
> QLoRA = LoRA + quantification 4-bit du modele de base. W_0 est stocke en NF4 (4 bits), dequantifie en BF16 pour le calcul, puis A et B sont entraines normalement en BF16. Permet de fine-tuner des modeles 70B sur un seul A100 80GB.

**Q : LoRA vs full fine-tuning : quand choisir lequel ?**
> LoRA si : ressources limitees, dataset petit, besoin de multiple adapteurs (hot-swap), domaine proche du pre-entrainement. Full FT si : beaucoup de GPU disponibles, dataset tres large, tache tres eloignee du pre-entrainement.

**Q : Expliquer le pipeline RLHF.**
> 3 etapes : (1) SFT = fine-tuner sur des demos humaines ; (2) Reward Model = entrainer un classificateur de preference sur des paires (w > l) avec la loss BT ; (3) PPO = optimiser la policy avec le RM comme signal de recompense, + contrainte KL pour ne pas trop devier du SFT.

**Q : DPO vs RLHF : avantages et inconvenients ?**
> DPO est mathematiquement equivalent a RLHF optimal mais sans reward model ni PPO. Avantage : simple, stable, un seul modele a entrainer. Inconvenient : moins flexible, pas de reward model reutilisable, peut overfit sur les paires si dataset petit.

**Q : Qu'est-ce que le reward hacking ?**
> Le modele apprend a maximiser le score du reward model sans reellement s'ameliorer. Ex : reponses tres longues, repetitives, sycophantiques. Solutions : ensemble de reward models, regularisation KL forte, diversite des evaluateurs.

**Q : Qu'est-ce que le catastrophic forgetting et comment l'eviter ?**
> Quand on fine-tune sur une nouvelle tache, le modele oublie les connaissances generales. Solutions : LoRA (W_0 reste intact), EWC (penaliser les changements sur les poids importants), learning rate tres faible, replay de donnees generales.
